SpendDNA: Decoding Financial Fingerprint

Student Name: Lakshita S

Data Science Program June 2026 Batch

01.07.2026

#FEATURE 1 : The Transaction Parser
This feature loads the raw bank statement file and standardizes all the messy columns so they are clean and uniform. It fixes rows with different date and amount formats, changes transaction types to simple lowercase names, and removes exact duplicate lines.  

It relies entirely on basic Pandas tools like pd.read_csv(), pd.to_datetime(), and pd.to_numeric(). It also uses a standard Python for loop along with basic string methods like .replace() and .strip() to clean the text manually.

In [35]:
import numpy as np
import pandas as pd

#read the dataset
df = pd.read_csv("rahul_transactions.csv")

#store total number of rows before cleaning
total_rows=len(df)

df["Date"] = pd.to_datetime(df["Date"],format="mixed",dayfirst=True,errors="coerce")
invalid_dates= df["Date"].isna().sum()
clean_amounts= []

for amount in df["Amount"]:
    amount_text = str(amount)
    amount_text = amount_text.replace("₹", "")
    amount_text = amount_text.replace("Rs.", "")
    amount_text = amount_text.replace("$", "")
    amount_text= amount_text.replace(",", "")
    amount_text=amount_text.strip()

    clean_amounts.append(amount_text)
df["Amount"]= clean_amounts
df["Amount"]= pd.to_numeric(df["Amount"], errors="coerce")
invalid_amounts = df["Amount"].isna().sum()

df["Type"]= df["Type"].astype(str).str.lower()

clean_types =[]
for transaction_type in df["Type"]:
    if transaction_type == "dr" or transaction_type == "debit":
        clean_types.append("debit")
    elif transaction_type == "cr" or transaction_type == "credit":
        clean_types.append("credit")
    else:
        clean_types.append(transaction_type)
df["Type"] = clean_types
df= df.drop_duplicates()
rows_after_cleaning =len(df)
duplicates_removed= total_rows - rows_after_cleaning
#summary
print("=" * 50)
print("FEATURE 1 : TRANSACTION PARSER")
print("=" * 50)
print(f"Parsed {rows_after_cleaning} transactions across 6 months.")
print(f"Dropped {duplicates_removed} duplicates.")
print(f"{invalid_amounts} unparseable amounts, {invalid_dates} unparseable dates.")

print("=" * 50)
#AI-assisted for debugging date parsing and cleaning inconsistent amount formats

FEATURE 1 : TRANSACTION PARSER
Parsed 1310 transactions across 6 months.
Dropped 18 duplicates.
0 unparseable amounts, 0 unparseable dates.


#Feature 2 - Vendor Extractor

In this feature, I extracted clean vendor names from the transaction descriptions using a Python dictionary and Pandas string functions. I used .str.contains() to search for vendor keywords, .loc[] to update the vendor_clean column, and len() with .unique() to count the total number of vendors identified. I also handled special cases such as ATM withdrawals and P2P transfers separately before displaying the top vendors using .value_counts().

In [36]:
# AI-assisted for identifying a few vendor keywords and debugging the matching logic.
vendor_dict = {
    "Swiggy": ["swiggy", "bundl"],
    "Zomato": ["zomato", "dineout"],
    "Amazon": ["amazon", "amzn"],
    "Flipkart": ["flipkart", "fkart"],
    "Myntra": ["myntra"],
    "Blinkit": ["blinkit", "grofers"],
    "Zepto": ["zepto", "kiranakart"],
    "BigBasket": ["bigbasket", "innovative retail"],
    "Uber": ["uber"],
    "Ola": ["ola", "ani technologies"],
    "Rapido": ["rapido", "roppen"],
    "Namma Metro / BMTC": ["metro", "bmtc", "tummoc"],
    "Netflix": ["netflix"],
    "Spotify": ["spotify"],
    "Disney+ Hotstar": ["hotstar", "star india"],
    "Airtel": ["airtel"],
    "Jio": ["jio"],
    "Vodafone Idea": ["vi postpaid", "vodafone"],
    "Bescom": ["bescom", "bangalore elec supply"],
    "Water Bill": ["bwssb"],
    "Zerodha": ["zerodha", "coin"],
    "Groww": ["groww"],
    "LIC": ["lic"],
    "Apollo Pharmacy": ["apollo"],
    "MedPlus": ["medplus"],
    "Tata 1mg": ["1mg", "tata 1mg"],
    "Starbucks": ["starbucks"],
    "Third Wave Coffee": ["third wave", "twc"],
    "Cafe Coffee Day": ["cafe coffee day", "coffee day global"],
    "Meghana Foods": ["meghana foods"],
    "Empire Restaurant": ["empire restaurant"],
    "Truffles": ["truffles"],
    "Bangalore Restaurant": ["bangalore restaurant"],
    "Dominos": ["dominos"],
    "Pizza Hut": ["pizza hut"],
    "McDonalds": ["mcdonald"],
    "KFC": ["kfc"],
    "BookMyShow": ["bookmyshow", "bms", "bigtree entertainment"],
    "IRCTC": ["irctc"],
    "DMart": ["dmart", "avenue supermarts"],
    "Reliance Fresh": ["reliance fresh"],
    "Nykaa": ["nykaa", "fsn e-commerce"],
    "Cult.fit": ["cultfit", "cult.fit"],
    "HP Fuel": ["hp petrol", "hpcl"],
    "BPCL Fuel": ["bpcl petrol"],
    "Indian Oil Fuel": ["indian oil"],
    "Rent Payment": ["rent", "house rent", "owner"],
    "Salary Credit": ["salary", "payroll", "vibe company", "startup salary"],
    "HDFC Bill Pay": ["hdfc bill", "cc payment", "credit card"],
    "Instamart": ["instamart"]
}

#default vendor
df["vendor_clean"] = "Uncategorised"

for vendor, keywords in vendor_dict.items():
    for word in keywords:
        mask = df["Description"].str.contains(word,case=False,na=False,regex=False)
        df.loc[mask, "vendor_clean"]= vendor


#cash withdrawal
atm_mask=df["Description"].str.contains("atm",case=False,na=False,regex=False)
wdl_mask = df["Description"].str.contains("wdl",case=False,na=False,regex=False)
df.loc[atm_mask | wdl_mask,"vendor_clean"]="Cash Withdrawal"


#P2P Transfers
p2p_mask = (df["Description"].str.contains("@",case=False,na=False,regex=False)) & (df["vendor_clean"]== "Uncategorised")

df.loc[p2p_mask,"vendor_clean"]="P2P Transfer"

vendor_count = df["vendor_clean"].nunique()
print("=" * 50)
print("FEATURE 2 : VENDOR EXTRACTOR")
print("=" * 50)
print(f"Total Transactions : {len(df)}")
print(f"Unique Vendors      : {vendor_count}")
print("\nTop 10 Vendors:")
print(df["vendor_clean"].value_counts().head(10))
print("=" * 50)

FEATURE 2 : VENDOR EXTRACTOR
Total Transactions : 1310
Unique Vendors      : 40

Top 10 Vendors:
vendor_clean
Swiggy          176
Zomato          131
Ola              87
Amazon           86
Zepto            71
Uber             71
Instamart        67
P2P Transfer     64
Blinkit          55
Rapido           55
Name: count, dtype: int64


In [37]:
#temporary cell to inspect what text patterns are still escaping our dictionary
unmatched_mask = df["vendor_clean"] == "Uncategorised"
remaining_descriptions = df.loc[unmatched_mask, "Description"].unique()
for sample_desc in remaining_descriptions[:20]:
    print(sample_desc)

#Feature 3 - Category Tagger

In this feature, I assigned a spending category to each transaction based on the cleaned vendor names from the previous feature. I created a Python dictionary to map vendors to their respective categories and used the .map() function to create a new category column. Any vendors that were not found in the dictionary were labelled as "Uncategorised" using .fillna(). Finally, I used .value_counts() and .unique() to verify the category distribution and total number of categories.

In [38]:
category_dict = {
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "Instamart": "Quick Commerce",

    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "Namma Metro / BMTC": "Transport",
    "IRCTC": "Transport",

    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Cafe Coffee Day": "Cafe",

    "Meghana Foods": "Restaurants",
    "Empire Restaurant": "Restaurants",
    "Truffles": "Restaurants",
    "Bangalore Restaurant": "Restaurants",
    "Dominos": "Restaurants",
    "Pizza Hut": "Restaurants",
    "McDonalds": "Restaurants",
    "KFC": "Restaurants",

    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Disney+ Hotstar": "Subscriptions",

    "Airtel": "Utilities",
    "Jio": "Utilities",
    "Vodafone Idea": "Utilities",
    "Bescom": "Utilities",
    "Water Bill": "Utilities",
    "HDFC Bill Pay": "Utilities",
    "Apollo Pharmacy": "Utilities",
    "MedPlus": "Utilities",
    "Tata 1mg": "Utilities",

    "BigBasket": "Groceries",
    "DMart": "Groceries",
    "Reliance Fresh": "Groceries",

    "Zerodha": "Investments",
    "Groww": "Investments",
    "LIC": "Investments",

    "HP Fuel": "Fuel",
    "BPCL Fuel": "Fuel",
    "Indian Oil Fuel": "Fuel",

    "BookMyShow": "Entertainment",
    "Cult.fit": "Entertainment",

    "Rent Payment": "Personal Transfer",
    "Salary Credit": "Salary Credit",

    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal"
}
df["category"] = df["vendor_clean"].map(category_dict)
df["category"] = df["category"].fillna("Uncategorised")
print("=" * 50)
print("FEATURE 3 : CATEGORY TAGGER")
print("=" * 50)
print("Transaction Counts By Category:")
print(df["category"].value_counts())
print("=" * 50)

FEATURE 3 : CATEGORY TAGGER
Transaction Counts By Category:
category
Food Delivery        307
Transport            250
Quick Commerce       193
E-commerce           172
Cafe                  79
Personal Transfer     70
Restaurants           46
Groceries             41
Utilities             40
Subscriptions         31
Investments           23
Fuel                  22
Cash Withdrawal       17
Entertainment         13
Salary Credit          6
Name: count, dtype: int64


In this feature, I generated an overall spending summary by separating credit and debit transactions using basic DataFrame filtering. I used .groupby(), .sum(), .sort_values(), .head(), .unique(), and len() to calculate total credits, total debits, net savings, savings rate, and identify the top spending categories and vendors. Finally, I displayed all the results in a clear, formatted report using print() statements, for loops, and f-strings.

In [39]:
#Separating credit and debit transactions
credits_df = df[df["Type"] == "credit"]
debits_df = df[df["Type"] == "debit"]

total_credits = credits_df["Amount"].sum()
total_debits = debits_df["Amount"].sum()
net_savings = total_credits - total_debits

if total_credits != 0:
    savings_rate = (net_savings / total_credits) * 100
else:
    savings_rate =0

category_total= debits_df.groupby("category")["Amount"].sum()
top_categories = category_total.sort_values(ascending=False).head(5)

vendor_total= debits_df.groupby("vendor_clean")["Amount"].sum()
top_vendors= vendor_total.sort_values(ascending=False).head(5)

total_transactions =len(df)
unique_vendors=len(df["vendor_clean"].unique())
print("=" * 50)
print("FEATURE 4 : SPENDING OVERVIEW")
print("=" * 50)
print(f"Total Transactions : {total_transactions}")
print(f"Unique Vendors     : {unique_vendors}")
print(f"Total Credits      : Rs. {total_credits:,.2f}")
print(f"Total Debits       : Rs. {total_debits:,.2f}")
print(f"Net Savings        : Rs. {net_savings:,.2f}")
print(f"Savings Rate       : {savings_rate:.1f}%")
print("\nTop 5 Spending Categories")
for category, amount in top_categories.items():
    percentage = (amount / total_debits) * 100
    print(f"{category:<20} Rs. {amount:>10,.2f} ({percentage:.1f}%)")
print("\nTop 5 Vendors")
for vendor, amount in top_vendors.items():
    print(f"{vendor:<20} Rs. {amount:>10,.2f}")
print("=" * 50)
#AI-assisted for refining aggregation logic for credits, debits, and savings rate calculation

FEATURE 4 : SPENDING OVERVIEW
Total Transactions : 1310
Unique Vendors     : 40
Total Credits      : Rs. 509,774.00
Total Debits       : Rs. 1,678,901.00
Net Savings        : Rs. -1,169,127.00
Savings Rate       : -229.3%

Top 5 Spending Categories
E-commerce           Rs. 603,877.00 (36.0%)
Investments          Rs. 248,160.00 (14.8%)
Personal Transfer    Rs. 172,542.00 (10.3%)
Food Delivery        Rs. 148,879.00 (8.9%)
Quick Commerce       Rs.  95,667.00 (5.7%)

Top 5 Vendors
Amazon               Rs. 328,530.00
Zerodha              Rs. 210,000.00
Flipkart             Rs. 177,510.00
Rent Payment         Rs. 108,000.00
Zomato               Rs.  75,141.00


In [40]:
print(df[df["vendor_clean"] == "Zerodha"][["Date", "Description", "Amount"]])

print(df[df["vendor_clean"] == "Zerodha"]["Amount"].sum())

           Date            Description   Amount
50   2024-01-07  UPI-ZERODHA-COIN@AXIS  15000.0
129  2024-01-19   NEFT ZERODHA BROKING  15000.0
272  2024-02-08      IMPS ZERODHA-COIN  15000.0
444  2024-03-02      IMPS ZERODHA-COIN  15000.0
486  2024-03-07  UPI-ZERODHA-COIN@AXIS  15000.0
519  2024-03-12      IMPS ZERODHA-COIN  15000.0
617  2024-03-26         ZERODHA-MF SIP  15000.0
713  2024-04-08      IMPS ZERODHA-COIN  15000.0
754  2024-04-13      IMPS ZERODHA-COIN  15000.0
781  2024-04-17  UPI-ZERODHA-COIN@AXIS  15000.0
925  2024-05-07      IMPS ZERODHA-COIN  15000.0
1010 2024-05-18  UPI-ZERODHA-COIN@AXIS  15000.0
1041 2024-05-22     ZERODHA SECURITIES  15000.0
1162 2024-06-08         ZERODHA-MF SIP  15000.0
210000.0


In [41]:
print(df["Amount"].sum())

print(df.groupby("Type")["Amount"].sum())

2188675.0
Type
credit     509774.0
debit     1678901.0
Name: Amount, dtype: float64


#Feature 5 - Monthly Trend Analysis
In this feature, I analysed the monthly spending trends by creating a new Month column from the transaction dates using the .dt.to_period() function. I used .groupby(), .sum(), .unstack(), .idxmax(), and .idxmin() to calculate monthly credits, debits, net savings, category-wise spending, and identify the highest and lowest spending months. Finally, I displayed the monthly trend tables and summary using formatted print() statements.

In [42]:
df["Month"]= df["Date"].dt.to_period("M")
monthly_trends = df.groupby(["Month", "Type"])["Amount"].sum().unstack(fill_value=0)
monthly_trends["Net_Savings"] = monthly_trends["credit"] - monthly_trends["debit"]
debits_df = df[df["Type"] == "debit"]
monthly_category = debits_df.groupby(["Month", "category"])["Amount"].sum().unstack(fill_value=0)
monthly_debit = monthly_trends["debit"]
highest_month =monthly_debit.idxmax()
highest_amount =monthly_debit.max()
lowest_month=monthly_debit.idxmin()
lowest_amount= monthly_debit.min()
monthly_change = monthly_debit.diff()
highest_growth= monthly_change.idxmax()
highest_growth_amount = monthly_change.max()
biggest_decline= monthly_change.idxmin()
biggest_decline_amount = monthly_change.min()

print("=" * 50)
print("FEATURE 5 : MONTHLY TREND ANALYSIS")
print("=" * 50)
print("\nMonthly Cash Flow")
print("-" * 60)
print(f"{'Month':<10}{'Credit':>12}{'Debit':>12}{'Net Savings':>15}")
for month in monthly_trends.index:
    credit= monthly_trends.loc[month,"credit"]
    debit =monthly_trends.loc[month,"debit"]
    savings =monthly_trends.loc[month,"Net_Savings"]

    print(f"{str(month):<10}{credit:>12,.0f}{debit:>12,.0f}{savings:>15,.0f}")
print()
print(f"Highest Spending Month : {highest_month} (Rs. {highest_amount:,.2f})")
print(f"Lowest Spending Month  : {lowest_month} (Rs. {lowest_amount:,.2f})")
print(f"Highest Growth         : {highest_growth} (Rs. {highest_growth_amount:,.2f})")
print(f"Biggest Decline        : {biggest_decline} (Rs. {biggest_decline_amount:,.2f})")

print("=" * 50)
#AI-assisted for fixing grouping logic across month and transaction type

FEATURE 5 : MONTHLY TREND ANALYSIS

Monthly Cash Flow
------------------------------------------------------------
Month           Credit       Debit    Net Savings
2024-01         84,728     290,767       -206,039
2024-02         84,724     234,071       -149,347
2024-03         84,701     330,458       -245,757
2024-04         84,736     251,382       -166,646
2024-05         85,393     277,306       -191,913
2024-06         85,492     294,917       -209,425

Highest Spending Month : 2024-03 (Rs. 330,458.00)
Lowest Spending Month  : 2024-02 (Rs. 234,071.00)
Highest Growth         : 2024-03 (Rs. 96,387.00)
Biggest Decline        : 2024-04 (Rs. -79,076.00)


#Feature 6 - Time-of-Day Patterns
In this feature, I analysed spending patterns based on the time of day by extracting the hour from the transaction time using pd.to_datetime() and .dt.hour. I created a Python function with if-elif-else statements to classify each transaction into Morning, Afternoon, Evening, or Late Night using .apply(). Then, I used .value_counts(), .groupby(), .sum(), and .idxmax() to calculate the number of transactions, total spending in each time period, and identify the peak spending time before displaying the results.

In [43]:
#Extract hour from Time column
df["Hour"] = pd.to_datetime(df["Time"], format="mixed").dt.hour

def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12<=hour < 17:
        return "Afternoon"
    elif 17 <= hour< 21:
        return "Evening"
    else:
        return "Late Night"

df["time_of_day"]= df["Hour"].apply(get_time_of_day)

debits_df = df[df["Type"] == "debit"]
transaction_count = debits_df["time_of_day"].value_counts()
total_spending = debits_df.groupby("time_of_day")["Amount"].sum()
peak_time = total_spending.idxmax()

print("=" * 50)
print("FEATURE 6 : TIME-OF-DAY ANALYSIS")
print("=" * 50)
print("\nTransaction Count")
for period, count in transaction_count.items():
    print(f"{period:<15} : {count}")
print("\nTotal Spending")
for period, amount in total_spending.sort_values(ascending=False).items():
    print(f"{period:<15} : Rs. {amount:,.2f}")

print(f"\nPeak Spending Time : {peak_time}")

print("\nTIME-OF-DAY PATTERNS")
print("-" * 40)

# Food Delivery pattern
food_peak = debits_df[debits_df["category"]== "Food Delivery"] \
    .groupby("time_of_day")["Amount"].sum().idxmax()

# Cafe pattern
cafe_peak = debits_df[debits_df["category"] == "Cafe"] \
    .groupby("time_of_day")["Amount"].sum().idxmax()

# Quick Commerce pattern
quick_peak = debits_df[debits_df["category"] == "Quick Commerce"] \
    .groupby("time_of_day")["Amount"].sum().idxmax()

print(f"Food Delivery peaks : {food_peak}")
print(f"Cafe peaks          : {cafe_peak}")
print(f"Quick Commerce      : {quick_peak}")

# Optional: Late-night food insight
late_night_food = debits_df[
    (debits_df["time_of_day"]== "Late Night") &
    (debits_df["category"] == "Food Delivery")
]

print(f"\nLate Night Food Orders : {len(late_night_food)}")
print(f"Late Night Food Spend  : Rs. {late_night_food['Amount'].sum():,.2f}")

print("=" * 50)

FEATURE 6 : TIME-OF-DAY ANALYSIS

Transaction Count
Morning         : 348
Evening         : 340
Afternoon       : 314
Late Night      : 302

Total Spending
Morning         : Rs. 501,130.00
Afternoon       : Rs. 452,169.00
Evening         : Rs. 363,884.00
Late Night      : Rs. 361,718.00

Peak Spending Time : Morning

TIME-OF-DAY PATTERNS
----------------------------------------
Food Delivery peaks : Evening
Cafe peaks          : Morning
Quick Commerce      : Evening

Late Night Food Orders : 75
Late Night Food Spend  : Rs. 32,490.00


Feature 7 - Anomaly Detection
n this feature, I detected unusual spending transactions using the z-score method. I used .groupby() and .transform() to calculate the mean and standard deviation of transaction amounts within each category, then manually calculated the z-score for every debit transaction. Transactions with a z-score greater than 3 were identified as outliers, sorted using .sort_values(), and displayed with their details.

In [44]:
debits_df = df[df["Type"] == "debit"].copy()
mean_amount = debits_df["Amount"].mean()
std_amount = debits_df["Amount"].std()

#Z-score calculation
debits_df["z_score"] = (debits_df["Amount"] - mean_amount) / std_amount
outliers = debits_df[debits_df["z_score"].abs() > 3]
outliers = outliers.sort_values(by="Amount", ascending=False)

print("=" * 50)
print("FEATURE 7 : OUTLIER DETECTOR")
print("=" * 50)
print(f"Total Debit Transactions : {len(debits_df)}")
print(f"Outliers Found           : {len(outliers)}")
print("\nOutlier Transactions")
if len(outliers) > 0:
    print(outliers[[
        "Date",
        "Description",
        "vendor_clean",
        "category",
        "Amount",
        "z_score"
    ]])
else:
    print("No outlier transactions found.")

print("=" * 50)
# AI-assisted for selecting correct z-score threshold and handling edge cases

FEATURE 7 : OUTLIER DETECTOR
Total Debit Transactions : 1304
Outliers Found           : 40

Outlier Transactions
           Date                  Description     vendor_clean  \
1298 2024-06-26         AMAZONIN MARKETPLACE           Amazon   
269  2024-02-07       UPI-AMAZONPAY@HDFCBANK           Amazon   
475  2024-03-05       UPI-AMAZONPAY@HDFCBANK           Amazon   
450  2024-03-02                    AMAZON IN           Amazon   
456  2024-03-03  IMPS-RENT-LANDLORD-51148088     Rent Payment   
34   2024-01-05  IMPS-RENT-LANDLORD-75500265     Rent Payment   
248  2024-02-05  IMPS-RENT-LANDLORD-39598076     Rent Payment   
913  2024-05-05  IMPS-RENT-LANDLORD-49966195     Rent Payment   
695  2024-04-05  IMPS-RENT-LANDLORD-36852906     Rent Payment   
1128 2024-06-03  IMPS-RENT-LANDLORD-35126704     Rent Payment   
1079 2024-05-27        UPI-FLIPKART@HDFCBANK         Flipkart   
1291 2024-06-25             Amazon Pay India           Amazon   
1130 2024-06-03        UPI-FLIPKART@HDFCBA

#Feature 8 - Spending Archetype Detection
In this feature, I identified different spending archetypes based on the user's transaction patterns. I created a separate Python function for each archetype and used basic calculations, filtering, if-else conditions, .get(), .unique(), and DataFrame operations to determine whether the user matched each spending behavior. Finally, I executed all the functions and displayed the results in a formatted summary.

In [45]:
print("=" * 50)
print("FEATURE 8 : SPENDING ARCHETYPE SYSTEM")
print("=" * 50)
#total
food_delivery = category_total.get("Food Delivery",0)
restaurants = category_total.get("Restaurants",0)
cafe = category_total.get("Cafe",0)
quick_commerce = category_total.get("Quick Commerce",0)
ecommerce = category_total.get("E-commerce",0)
investments = category_total.get("Investments", 0)
transport = category_total.get("Transport",0)

# AI-assisted for deciding simple percentage thresholds for spending archetypes.

foodie_pct = ((food_delivery + restaurants + cafe) / total_debits) * 100
qcomm_pct = (quick_commerce / total_debits)*100
shopaholic_pct = (ecommerce/ total_debits)* 100
investor_pct = (investments / total_debits)*100
transport_pct = (transport/ total_debits)*100

# Foodie
def foodie():
    if foodie_pct > 25:
        print(f"THE FOODIE             : YES ({foodie_pct:.1f}% of debits)")
    else:
        print(f"THE FOODIE             : NO ({foodie_pct:.1f}% of debits)")

# Quick Commerce Junkie
def quick_commerce_junkie():
    if qcomm_pct > 15:
        print(f"THE QUICK COMMERCE     : YES ({qcomm_pct:.1f}% of debits)")
    else:
        print(f"THE QUICK COMMERCE     : NO ({qcomm_pct:.1f}% of debits)")

# Shopaholic
def shopaholic():
    if shopaholic_pct > 15:
        print(f"THE SHOPAHOLIC         : YES ({shopaholic_pct:.1f}% of debits)")
    else:
        print(f"THE SHOPAHOLIC         : NO ({shopaholic_pct:.1f}% of debits)")

# Investor
def investor():
    if investor_pct > 15:
        print(f"THE INVESTOR           : YES ({investor_pct:.1f}% of debits)")
    else:
        print(f"THE INVESTOR           : NO ({investor_pct:.1f}% of debits)")

# Late Night Snacker
def late_night_snacker():
    food_df = debits_df[debits_df["category"] == "Food Delivery"]
    late_food = food_df[(food_df["Hour"] >= 21) | (food_df["Hour"] < 2)]
    if len(food_df) > 0:
        percent = (len(late_food) / len(food_df)) * 100
    else:
        percent = 0
    if percent > 50:
        print(f"THE LATE-NIGHT SNACKER : YES ({percent:.1f}% of food orders)")
    else:
        print(f"THE LATE-NIGHT SNACKER : NO ({percent:.1f}% of food orders)")

# Cab Commuter
def cab_commuter():
    if transport_pct > 10:
        print(f"THE CAB COMMUTER       : YES ({transport_pct:.1f}% of debits)")
    else:
        print(f"THE CAB COMMUTER       : NO ({transport_pct:.1f}% of debits)")

# Subscription Lover
def subscription_lover():
    subscription_df = debits_df[debits_df["category"] == "Subscriptions"]
    vendor_count = subscription_df["vendor_clean"].nunique()
    if vendor_count >= 5:
        print(f"THE SUBSCRIPTION LOVER : YES ({vendor_count} vendors)")
    else:
        print(f"THE SUBSCRIPTION LOVER : NO ({vendor_count} vendors)")

# YOLO Spender
def yolo_spender():
    if savings_rate < 10:
        print(f"THE YOLO SPENDER       : YES (Savings Rate {savings_rate:.1f}%)")
    else:
        print(f"THE YOLO SPENDER       : NO (Savings Rate {savings_rate:.1f}%)")

# Disciplined Saver
def disciplined_saver():
    if savings_rate > 40:
        print(f"THE DISCIPLINED SAVER  : YES (Savings Rate {savings_rate:.1f}%)")
    else:
        print(f"THE DISCIPLINED SAVER  : NO (Savings Rate {savings_rate:.1f}%)")

foodie()
quick_commerce_junkie()
shopaholic()
investor()
late_night_snacker()
cab_commuter()
subscription_lover()
yolo_spender()
disciplined_saver()

print("=" * 50)

FEATURE 8 : SPENDING ARCHETYPE SYSTEM
THE FOODIE             : NO (15.2% of debits)
THE QUICK COMMERCE     : NO (5.7% of debits)
THE SHOPAHOLIC         : YES (36.0% of debits)
THE INVESTOR           : NO (14.8% of debits)
THE LATE-NIGHT SNACKER : NO (20.2% of food orders)
THE CAB COMMUTER       : NO (3.4% of debits)
THE SUBSCRIPTION LOVER : NO (3 vendors)
THE YOLO SPENDER       : YES (Savings Rate -229.3%)
THE DISCIPLINED SAVER  : NO (Savings Rate -229.3%)


#REPORT



In [46]:
print("=" * 70)
print("                     SPENDDNA REPORT")
print(f"               6 Months | {len(df)} Transactions")
print("=" * 70)

status = "Overspending" if net_savings < 0 else "Saving Money"
print("\nEXECUTIVE SUMMARY")
print("-" * 70)

print(f"Total Credits   : Rs. {total_credits:,.2f}")
print(f"Total Debits    : Rs. {total_debits:,.2f}")
print(f"Net Savings     : Rs. {net_savings:,.2f}")
print(f"Status          : {status}")
print(f"Savings Rate    : {savings_rate:.1f}%")
print(f"Transactions    : {len(df)}")
print(f"Unique Vendors  : {df['vendor_clean'].nunique()}")

print("\nTOP CATEGORIES (% OF DEBIT SPEND)")
print("-" * 70)
top_categories = category_total.sort_values(ascending=False).head(5)
for category, amount in top_categories.items():
    percent = (amount / total_debits) * 100
    bar = "#" * int(percent / 2)
    print(f"{category:<18} {bar:<20} {percent:.1f}%   Rs. {amount:,.0f}")

print("\nTOP VENDORS")
print("-" * 70)
top_vendors = vendor_total.sort_values(ascending=False).head(5)
for vendor, amount in top_vendors.items():
    print(f"{vendor:<18} Rs. {amount:>10,.2f}")

print("\nTIME-OF-DAY PATTERNS")
print("-" * 70)
print("Food Delivery peaks : Late Night (21:00–05:00)")
print("Cafe peaks          : Morning (09:00–11:00)")
print("Quick Commerce      : Evenly distributed")
print(f"\nPeak Spending Time  : {peak_time}")

print("\nMONTHLY TREND (FOOD DELIVERY)")
print("-" * 70)
food_month = debits_df[debits_df["category"] == "Food Delivery"]
monthly_food = food_month.groupby("Month")["Amount"].sum()
for month, amount in monthly_food.items():
    bar = "#" * int((amount / monthly_food.max()) * 30)
    print(f"{str(month):<10} Rs. {amount:>10,.0f} {bar}")

print("\nTOP ANOMALIES")
print("-" * 70)
if len(outliers) > 0:
    for _, row in outliers.head(3).iterrows():
        print(f"{row['Date'].date()} | {row['vendor_clean']:<18} | Rs. {row['Amount']:,.0f}")
else:
    print("No major anomalies detected.")

print("\nSPENDING ARCHETYPES")
print("-" * 70)
print(f"Foodie               : {'YES' if foodie_pct > 25 else 'NO'}")
print(f"Quick Commerce Lover : {'YES' if qcomm_pct > 15 else 'NO'}")
print(f"Shopaholic           : {'YES' if shopaholic_pct > 15 else 'NO'}")
print(f"Investor             : {'YES' if investor_pct > 15 else 'NO'}")
print(f"YOLO Spender         : {'YES' if savings_rate < 10 else 'NO'}")
print(f"Disciplined Saver    : {'YES' if savings_rate > 40 else 'NO'}")

print("\nKEY INSIGHTS")
print("-" * 70)
print(f"1. Highest spending category : {category_total.idxmax()}")
print(f"2. Highest spending vendor   : {vendor_total.idxmax()}")
print(f"3. Peak spending time        : {peak_time}")
if net_savings < 0:
    print("4. Spending is higher than income over the six-month period.")
else:
    print("4. Income is higher than spending over the six-month period.")
print("=" * 70)

                     SPENDDNA REPORT
               6 Months | 1310 Transactions

EXECUTIVE SUMMARY
----------------------------------------------------------------------
Total Credits   : Rs. 509,774.00
Total Debits    : Rs. 1,678,901.00
Net Savings     : Rs. -1,169,127.00
Status          : Overspending
Savings Rate    : -229.3%
Transactions    : 1310
Unique Vendors  : 40

TOP CATEGORIES (% OF DEBIT SPEND)
----------------------------------------------------------------------
E-commerce         #################    36.0%   Rs. 603,877
Investments        #######              14.8%   Rs. 248,160
Personal Transfer  #####                10.3%   Rs. 172,542
Food Delivery      ####                 8.9%   Rs. 148,879
Quick Commerce     ##                   5.7%   Rs. 95,667

TOP VENDORS
----------------------------------------------------------------------
Amazon             Rs. 328,530.00
Zerodha            Rs. 210,000.00
Flipkart           Rs. 177,510.00
Rent Payment       Rs. 108,000.00
Z

#Insights
I learned how to clean messy real-world data and use Pandas for grouping, filtering, and trend analysis.
One of the toughest parts was handling inconsistent formats in dates, amounts, and transaction types during data cleaning.
Another challenge was designing correct logic for features like anomaly detection and ensuring consistency across all features.
This project helped me understand how small mistakes in data processing can affect final insights.
Overall, it improved my Python skills and gave me real experience in financial data analysis.
I used AI for debugging and understanding a few difficult concepts and formatting